# QIF-Micro tutorial

Welcome to the QIF-Micro tutorial!  
This notebook provides an introduction to the Quantitative Information Flow (QIF) library for privacy analysis of microdata releases.

Let's start by importing the necessary libraries and modules from QIF-Micro.

In [1]:
from functools import partial

import numpy as np
import polars as pl
import scipy.sparse as sp

np.set_printoptions(linewidth=100)

# Import main QIF-Micro modules
from qif_micro import qif, mechanism, model, measure

# Import core data types
from qif_micro.qif.datatypes import Channel, Joint, ProbabDist, Strategy

In [2]:
type QIFDatatype = Channel | Joint | ProbabDist | Strategy
def inspect(qif_obj: QIFDatatype) -> QIFDatatype:
    return qif_obj.dist.toarray() if sp.issparse(qif_obj.dist) else qif_obj.dist

## QIF-Micro library overview

QIF-Micro is a comprehensive library for **Quantitative Information Flow (QIF)** privacy analysis. It measures information leakage in privacy-preserving data releases through mathematical channel models. The QIF-Micro library is composed of some modules, namely: ``qif``, ``mechanism``, ``model`` and ``measure``. We will explore each one of them with some examples.

## The QIF module

The `qif` module is the mathematical heart of QIF-Micro. It provides core operations for QIF analysis. 
It is implemented in terms of four **data types**: 
 
- **`ProbabDist`**: A probability distribution over some domain
- **`Channel`**: A stochastic matrix representing the probability that a system produces an observable output $y$ from a secret input $x$
- **`Joint`**: A joint probability distribution of secrets and observable outputs
- **`Strategy`**: A stochastic matrix representing the adversary's best strategy for each output

Each of these classes are wrappers around `numpy` and `scipy.sparse`, meaning that one can choose between a dense or sparse representation. The main purpose of these classes is to validate the construction of QIF objects: all values must be between 0 and 1; in the case of `ProbabDist` and `Joint`, the values must add up to 1; and, in the case of `Channel` and `Strategy`, the sum of the rows must be 1.

The four classes have two common fields `dist` and `is_slice`. By default, `is_slice` is set to `False`, but one can construct a slice of the data type by passing `is_slice=True`; this will disable the check for sum = 1.

Finally, except for `ProbabDist`, the underlying `dist` may be partitioned by columns.

Let's explore its main functions with a practical example.

### Understanding channels and distributions

A **channel** is a stochastic matrix where each row represents a possible secret value, and each column represents an observable output. Each entry `ch[i, j]` is the probability that the system produces the output $b_j$ given the secret input $a_i$.

A **prior distribution** `π` represents an adversary's belief about how likely each secret is before observing the output.

Let's create a simple example:

In [3]:
# Create a simple prior: 3 equally likely secrets
pi = ProbabDist(np.array([1/3, 1/3, 1/3]))
pi.dist

array([0.33333333, 0.33333333, 0.33333333])

In [4]:
# Create a simple channel (stochastic matrix)
# Each row sums to 1 (probabilities)
dist = np.array([
    [0.7, 0.2, 0.1],  # If secret is 0, output is mostly 0
    [0.9, 0.0, 0.1],  # If secret is 1, output is mostly 1
    [0.3, 0.7, 0.0]   # If secret is 2, output is mostly 2
], dtype=float)

ch = Channel(dist)
ch.dist

array([[0.7, 0.2, 0.1],
       [0.9, 0. , 0.1],
       [0.3, 0.7, 0. ]])

To save some memory, we can create a sparse channel by changing `np.array` to `csr_array` (from `scipy.sparse`)

In [5]:
# Create a simple channel (stochastic matrix)
# Each row sums to 1 (probabilities)
dist = sp.csr_array([
    [0.7, 0.2, 0.1],  # If secret is 0, output is mostly 0
    [0.9, 0.0, 0.1],  # If secret is 1, output is mostly 1
    [0.3, 0.7, 0.0]   # If secret is 2, output is mostly 2
], dtype=float)

ch = Channel(dist)
ch.dist

<Compressed Sparse Row sparse array of dtype 'float64'
	with 7 stored elements and shape (3, 3)>

Notice that only 7 elements were actually stored. To visualise the matrix when using a sparse array, use the method `toarray`:

In [6]:
ch.dist.toarray()

array([[0.7, 0.2, 0.1],
       [0.9, 0. , 0.1],
       [0.3, 0.7, 0. ]])

To help visualising the matrix representations, from now on we will use the `inspect` function defined at the top.

### Computing the joint distribution

The joint distribution combines the prior and the channel to give us the joint probability of secrets and outputs.  
This is computed as: $\texttt{joint[i, j]} := \pi\texttt{[i]}\; \texttt{ch[i, j]}$

In [7]:
# Compute the joint distribution
joint_dist = qif.joint(pi, ch)
inspect(joint_dist)

array([[0.23333333, 0.06666667, 0.03333333],
       [0.3       , 0.        , 0.03333333],
       [0.1       , 0.23333333, 0.        ]])

### Computing posterior probabilities

After observing an output, an adversary updates their belief about the secret using Bayes' rule. The posterior probability of the secret input $a_i$ given the observed output $b_j$ is computed as: $
\mathrm{Pr}(\mathrm{secret} = a_i \mid \mathrm{output} = b_j) := \texttt{joint[i, j]}\; \mathrm{Pr}(\mathrm{output} = a_j)
$

The posterior distributions can be obtained using `qif.hyper`, which takes either a prior and a channel, or a joint, and returns (i) the marginal distribution on outputs, which we call the outer distribution; and (ii) a channel in which each row is a posterior distribution given an output:

In [8]:
outer, posteriors = qif.hyper(joint_dist)
inspect(outer)

array([0.63333333, 0.3       , 0.06666667])

In [9]:
inspect(posteriors).T # Transpose so that columns are the outputs, our usual representation

array([[0.36842105, 0.22222222, 0.5       ],
       [0.47368421, 0.        , 0.5       ],
       [0.15789474, 0.77777778, 0.        ]])

### Computing vulnerability metrics

The **Bayes vulnerability** measures the adversary's probability of inferring the secret correctly in one try, if they adopt the optimal strategy: to guess the most likely secret given the output. Other vulnerability metrics can be modelled via gain functions (TODO).

In [10]:
adv_st = qif.strategy(joint_dist) # Takes a joint, or a prior + channel
inspect(adv_st).T # Transpose, to get outputs as columns

array([[0. , 0. , 0.5],
       [1. , 0. , 0.5],
       [0. , 1. , 0. ]])

We measure the vulnerability of the secret before and after the adversary observes an output (as an average over all outputs):

In [11]:
qif.measure.bayes.prior(pi), qif.measure.bayes.posterior(joint_dist)

(np.float64(0.3333333333333333), np.float64(0.5666666666666667))

## The mechanism module

The `mechanism` module provides pre-built privacy mechanisms that generate channels.  
These mechanisms add noise to data in controlled ways to limit information leakage. 

### Random response mechanism

The random response mechanism is a classic privacy technique for binary or categorical data. For instance, one may want to add  noise to a sensitive attribute `disability` before releasing a dataset. This can be constructed as follows, where $p$ is the probability of *preserving* the original value:

In [12]:
domain = [0, 1, 2]

rr = mechanism.random_response(
    p=0.7,  # 70% chance of truthful response, 30% random
    input_domain=domain,  # 3 possible values (e.g., no, maybe or yes)
    output_domain=domain
)

inspect(rr)

array([[0.7 , 0.15, 0.15],
       [0.15, 0.7 , 0.15],
       [0.15, 0.15, 0.7 ]])

Sometimes (perhaps due to memory constraints) we do not want to construct the entire matrix. We can get a slice, by changing the output domain:

In [13]:
input_domain = [0, 1, 2]
output_domain = [0, 2]

rr, row_labels, col_labels = mechanism.random_response(
    p=0.7,  # 70% chance of truthful response, 30% random
    input_domain=input_domain,  # 3 possible values (e.g., Yes/No/Maybe)
    output_domain=output_domain, # Only 2 possible values in the slice
    domain_size=3, # Required for slices
    return_labels=True
)

inspect(rr)

array([[0.7 , 0.15],
       [0.15, 0.7 ],
       [0.15, 0.15]])

In [14]:
row_labels, col_labels

(array([0, 2, 1]), array([0, 2]))

### The truncated geometric mechanism

The geometric mechanism adds distance-based noise to discrete-valued data. The probability of observing output $a$ given secret $b$ depends on the distance between them. Formally, the truncated geometric noise is defined as follows, for $0 < \alpha \leq 1$:

$$
\mathrm{Pr}(\mathrm{output} = b \mid \mathrm{secret} = a) := \frac{\alpha^{\vert a - b \vert}}{1 + \alpha}\, (1 - \alpha) 
\textbf{ if } \min \mathcal{B} < b < \max \mathcal{B} \textbf{ else } \frac{\alpha^{\vert a - b \vert}}{1 + \alpha}
$$

In [15]:
domain = [0, 1, 2]

tg = mechanism.geometric(
    alpha=1/3,
    input_domain=domain,
    output_domain=domain
)

inspect(tg)

array([[0.75      , 0.16666667, 0.08333333],
       [0.25      , 0.5       , 0.25      ],
       [0.08333333, 0.16666667, 0.75      ]])

As with random response, we can construct a slice of the truncated geometric mechanism. In this case, instead of informing the size of the domain, we must inform the minimum and maximum values in the output:

In [16]:
input_domain = [0, 1, 2]
output_domain = [0, 2]

tg, row_labels, col_labels = mechanism.geometric(
    alpha=1/3,
    input_domain=input_domain,
    output_domain=output_domain,
    domain_min=0, 
    domain_max=2,
    return_labels=True
)

inspect(tg)

array([[0.75      , 0.08333333],
       [0.25      , 0.25      ],
       [0.08333333, 0.75      ]])

In [17]:
row_labels, col_labels

(array([0, 1, 2]), array([0, 2]))

### Record-level mechanisms

To model privacy-preserving data-release pipelines, we need a privacy mechanism on the domain of records. This can be constructed by hand, by defining the domain of records and building a channel that maps records to (possibly different, sanitised) records. Nonetheless, the QIF-Micro library offers a helper function that constructs a record-level mechanism from attribute-level mechanisms such as random response and geometric noise:

Consider the following domain of records, for now assuming that records have length one:

In [18]:
domain_grade = [0, 1, 2] # Say, "A", "B" and "C"
domain_dis = [0, 1] # No or yes, for some disability

domain = [
    [{"grade": grade, "disability": dis}]
    for grade in domain_grade
    for dis in domain_dis
]

We will prepare some mechanism functions, that will later be invoked to construct the actual channels:

In [19]:
m_grade = partial(mechanism.geometric, alpha=1/3)
m_dis = partial(mechanism.random_response, p=1/2)

Now we can construct the mechanism from records to records, which applies random response to `disability` and geometric noise to `grade`:

In [20]:
m = mechanism.record(domain, disability=m_dis, grade=m_grade)
inspect(m)

array([[0.375     , 0.375     , 0.08333333, 0.08333333, 0.04166667, 0.04166667],
       [0.375     , 0.375     , 0.08333333, 0.08333333, 0.04166667, 0.04166667],
       [0.125     , 0.125     , 0.25      , 0.25      , 0.125     , 0.125     ],
       [0.125     , 0.125     , 0.25      , 0.25      , 0.125     , 0.125     ],
       [0.04166667, 0.04166667, 0.08333333, 0.08333333, 0.375     , 0.375     ],
       [0.04166667, 0.04166667, 0.08333333, 0.08333333, 0.375     , 0.375     ]])

Instead of a list of records, we can also use a polars DataFrame. In this case there are two mandatory attributes: `record_id` and `entry_id` (always 0 if records have length one). The name of the columns can be controlled via parameters `record_col` and `entry_col`.

In [21]:
n_records = len(domain_grade) * len(domain_dis)

domain = pl.DataFrame({
    "record_id":  [i for i in range(n_records)],
    "entry_id":   [0 for _ in range(n_records)],
    "grade":      np.repeat(domain_grade, len(domain_dis)),
    "disability": np.tile(domain_dis, len(domain_grade))
})

domain

record_id,entry_id,grade,disability
i64,i64,i64,i64
0,0,0,0
1,0,0,1
2,0,1,0
3,0,1,1
4,0,2,0
5,0,2,1


In [22]:
m = mechanism.record(domain, disability=m_dis, grade=m_grade)
inspect(m)

array([[0.375     , 0.375     , 0.08333333, 0.08333333, 0.04166667, 0.04166667],
       [0.375     , 0.375     , 0.08333333, 0.08333333, 0.04166667, 0.04166667],
       [0.125     , 0.125     , 0.25      , 0.25      , 0.125     , 0.125     ],
       [0.125     , 0.125     , 0.25      , 0.25      , 0.125     , 0.125     ],
       [0.04166667, 0.04166667, 0.08333333, 0.08333333, 0.375     , 0.375     ],
       [0.04166667, 0.04166667, 0.08333333, 0.08333333, 0.375     , 0.375     ]])

Finally, we can control the domains for each attribute-level mechanism, to construct only a slice of the record-level mechanism. In this case we must inform the size of the domain of sanitised records, or else the lib assumes we are constructing the whole channel:

In [23]:
m_grade = partial(m_grade, output_domain=[0, 2], domain_min=0, domain_max=2)
m_dis = partial(mechanism.random_response, p=1/2)
m = mechanism.record(domain, output_size=n_records, disability=m_dis, grade=m_grade)
inspect(m)

array([[0.375     , 0.375     , 0.        , 0.        , 0.04166667, 0.04166667],
       [0.375     , 0.375     , 0.        , 0.        , 0.04166667, 0.04166667],
       [0.125     , 0.125     , 0.        , 0.        , 0.125     , 0.125     ],
       [0.125     , 0.125     , 0.        , 0.        , 0.125     , 0.125     ],
       [0.04166667, 0.04166667, 0.        , 0.        , 0.375     , 0.375     ],
       [0.04166667, 0.04166667, 0.        , 0.        , 0.375     , 0.375     ]])

We reinforce that the above is just a matrix representation, but the zeros are not really there:

In [24]:
full_channel = n_records * n_records
stored = m.dist.nnz
full_channel, stored

(36, 24)

Or we can directly control the output domain of records:

In [25]:
output_domain = domain.filter(pl.col("grade") != 1)
m_grade = partial(mechanism.geometric, alpha=1/3)
m_dis = partial(mechanism.random_response, p=1/2)

m = mechanism.record(
    domain, output_domain, output_size=n_records, 
    disability=m_dis, grade=m_grade
)

inspect(m)

array([[0.375     , 0.375     , 0.        , 0.        , 0.04166667, 0.04166667],
       [0.375     , 0.375     , 0.        , 0.        , 0.04166667, 0.04166667],
       [0.125     , 0.125     , 0.        , 0.        , 0.125     , 0.125     ],
       [0.125     , 0.125     , 0.        , 0.        , 0.125     , 0.125     ],
       [0.04166667, 0.04166667, 0.        , 0.        , 0.375     , 0.375     ],
       [0.04166667, 0.04166667, 0.        , 0.        , 0.375     , 0.375     ]])

## The model and measure modules

The `model` module defines two main models `baseline` and `generic`, along with specialised models. Currently, the only specialised model available is `count_sum`. We will explore these models with some examples. Overall,

- The `baseline` model corresponds to an adversary who observes the real (de-identified) data
- The `generic` model is a flexible model that can be used to construct different scenarios, by passing record-level mechanisms
- The `count_sum` model corresponds to an adversary who observes the result of a count-sum (possibly grouped) query

The result of `baseline` is a `typing.BaselineModel`, which is always a joint distribution, plus optionally some labels. The result of any other model is a `typing.Model`, which is a pair `(Joint, Strategy`): the baseline joint and the adversary's strategy.

These models can then be passed onto `measure.linkage_risk` to assess the inference risk in the linkage attack.

**Note**: Currently, we assume that the adversary's goal is to reconstruct the target's original detailed record, except in the case of `count_sum`, in which we assume that the adversary's goal is to reconstruct the target's aggregated record.

### The baseline model

Consider the following dataset with some detailed data about consumers' transactions, and suppose that the adversary has some auxiliary information about the amount and category of one of the transactions made by their target:

In [26]:
dataset = pl.DataFrame({
    "owner_id": [0, 0, 1, 1, 2, 2, 2],
    "entry_id": [0, 1, 0, 1, 0, 1, 2],
    "amount":   [0, 2, 1, 1, 1, 0, 2],
    "category": [0, 0, 0, 0, 0, 1, 1]
})

dataset

owner_id,entry_id,amount,category
i64,i64,i64,i64
0,0,0,0
0,1,2,0
1,0,1,0
1,1,1,0
2,0,1,0
2,1,0,1
2,2,2,1


The adversary starts with some prior knowledge about how records are distributed. But, since in this case the adversary observes the real data, upon observing the dataset they replace their prior knowledge with the distribution of records in the dataset. This is why we do not need a proper prior on records here. We can thus construct the baseline model directly from the dataset:

In [27]:
hint = ["amount", "category"]
baseline_joint, hint_labels = model.baseline(dataset, hint, return_labels=True)
inspect(baseline_joint)

array([[0.16666667, 0.        , 0.        , 0.16666667, 0.        ],
       [0.        , 0.11111111, 0.11111111, 0.        , 0.11111111],
       [0.        , 0.        , 0.33333333, 0.        , 0.        ]])

The columns (hints) in the channel above corresponds to the following pairs of (amount, category):

In [28]:
hint_labels.sort("hint").collect(engine="streaming")

hint_label,hint
struct[2],u32
"{0,0}",0
"{0,1}",1
"{1,0}",2
"{2,0}",3
"{2,1}",4


Notice that, by linking their auxiliary information about the amount and category of one of the target's transactions with the observed dataset, the adversary in most cases (except for a transaction with cost $\$1$ and category 0) learns with certainty the target's record. The adversary's expected chance of correctly reconstructing the the target's original (secret) record can then be computed as follows:

In [29]:
measure.linkage_risk(baseline_joint)

np.float64(0.8888888888888888)

The baseline model also accepts a sequence of datasets, in which case it models a longitudinal scenario (e.g., monthly releases). To illustrate, consider that the consumers repeated the same transactions in the following month:

In [30]:
hint = ["amount", "category"]
baseline_joint = model.baseline([dataset, dataset], hint, opt_memory=False)
baseline_joint

Joint(dist=<Compressed Sparse Row sparse array of dtype 'float64'
	with 14 stored elements and shape (3, 13)>, is_complete=True)

Notice that there are 13 possible hints now, which are pairs of pairs (amount, category). The baseline model can optimise some pairs, in case one of the individual hints (first or second month) are already enough to uniquely identiy a record. The optimisation is enabled by default. The resulting dist is partitioned: the first partition contains the hints that have been optimised, and the second contains the combined hints:

In [31]:
hint = ["amount", "category"]
baseline_joint = model.baseline([dataset, dataset], hint)
baseline_joint

Joint(dist=[<Compressed Sparse Row sparse array of dtype 'float64'
	with 4 stored elements and shape (3, 4)>, <Compressed Sparse Row sparse array of dtype 'float64'
	with 4 stored elements and shape (3, 3)>], is_complete=True)

### The generic model

The generic model takes a prior distribution on records, the original dataset, the sanitised dataset and the record-level mechanism that was used to produce the sanitised dataset (naturally, the original and sanitised datasets must be compatible, according to the mechanism).

To illustrate, consider first a deterministic mechanism that suppresses the `amount` attribute in our running example:

In [32]:
def suppress(input_domain, output_domain = None, return_labels: bool = False):
    if output_domain is None: output_domain = ["*"]
    if len(output_domain) > 1: 
        raise ValueError("Output domain must have a single value")

    input_domain = np.unique(input_domain)
    n_rows = input_domain.shape[0]
    n_cols = 1

    dist = np.ones(shape=(n_rows, n_cols))
    ch = Channel(dist)
    
    return (ch, input_domain, output_domain) if return_labels else ch

Due to a current limitation in the implementation of the generic model, the original and sanitised records for the generic model must have the same types, so we have to make sure that we suppress `amount` by replacing it with a fixed value of the same type:

In [33]:
amount_dtype = dataset.collect_schema()["amount"]
suppress_expr = pl.lit(0).cast(amount_dtype).alias("amount")
sanitised_dataset = dataset.with_columns(suppress_expr)
sanitised_dataset

owner_id,entry_id,amount,category
i64,i64,i64,i64
0,0,0,0
0,1,0,0
1,0,0,0
1,1,0,0
2,0,0,0
2,1,0,1
2,2,0,1


We need to get the sub-domain of records that the adversary will consider as possible after observing the sanitised dataset. Since we only have records with length 2 and 3, we focus on them, fixing the original `category`. We assume that the domain of `amount`is `[0, 1, 2]`.

In [34]:
def extract_records(
    dataset: pl.DataFrame, 
    owner_col: str = "owner_id",
    entry_col: str = "entry_id"
) -> pl.DataFrame:
    dataset = dataset.sort(owner_col, entry_col)
    as_records = dataset.group_by(owner_col).agg(pl.all())
    return as_records.drop(owner_col).unique()

In [35]:
domain_amount = [0, 1, 2]

repeat = np.repeat(domain_amount, len(domain_amount)).tolist()
tile = np.tile(domain_amount, len(domain_amount)).tolist()
domain_amount_2 = [[a, b] for a, b in zip(repeat, tile)]

repeat = np.repeat(domain_amount_2, len(domain_amount), axis=0).tolist()
tile = np.tile(domain_amount, len(domain_amount_2)).tolist()
domain_amount_3 = [[*ab, c] for ab, c in zip(repeat, tile)]

original_records = extract_records(dataset)

length_2 = (
    original_records.filter(pl.col("amount").list.len() == 2)
    .drop("amount").unique()
    .with_columns(pl.lit(domain_amount_2).alias("amount"))
    .explode("amount")
)

length_3 = (
    original_records.filter(pl.col("amount").list.len() == 3)
    .drop("amount").unique()
    .with_columns(pl.lit(domain_amount_3).alias("amount"))
    .explode("amount")
)

input_domain = (
    pl.concat([length_2, length_3])
    .with_row_index("record_id")
    .explode(pl.exclude("record_id"))
)

record_attrs = ["entry_id", "amount", "category"]
output_domain = (
    input_domain
    .sort("record_id", "entry_id").group_by("record_id").agg(pl.all())
    .join(extract_records(sanitised_dataset), on=record_attrs)
    .explode(pl.exclude("record_id"))
)

Then we construct a prior on the (sub-)domain of records, and construct the mechanism:

In [36]:
n_records = input_domain["record_id"].n_unique()
dist = np.repeat(1 / n_records, n_records)
pi = ProbabDist(dist)

m_amount = partial(suppress, output_domain=[0])
m = mechanism.record(input_domain, output_domain, amount=m_amount)

As mentioned before, except for `baseline`, the models return a baseline joint and the adversary's strategy.

If you inspect the baseline joint at `adv_model[0]`, you will notice that it is exactly the same joint that we obtained from the baseline model, except that it now has a bunch of rows that are full of zeros, to align with the adversary's strategy, and columns and rows may be permutated.

In [37]:
adv_model = model.generic(pi, input_domain, m, dataset, sanitised_dataset, hint)
adv_model

(Joint(dist=<Compressed Sparse Row sparse array of dtype 'float64'
 	with 6 stored elements and shape (36, 5)>, is_complete=True),
 Channel(dist=<Compressed Sparse Column sparse array of dtype 'float64'
 	with 72 stored elements and shape (5, 36)>, is_complete=True))

which we can then pass to `measure.linkage_risk` to assess inference risk in the linkage attack (notice how risk has decreased):

In [38]:
measure.linkage_risk(adv_model)

np.float64(0.3333333333333333)

Naturally, we can also employ probabilistic mechanisms. For instance, what if, instead of suppressing `amount` altogether, we add noise to it?  
The following helper function applies a record-level mechanism to a dataset to produce a sanitised dataset:

In [39]:
def run_mechanism(
    dataset: pl.DataFrame, 
    domain_records: pl.DataFrame,
    m: Channel,
    owner_col: str = "owner_id",
    record_col: str = "record_id",
    entry_col: str = "entry_id"
) -> pl.DataFrame:
    p, rows, cols = m.dist.tocoo().data, *m.dist.tocoo().coords
    
    m_df = (
        pl.DataFrame({"row": rows, "col": cols, "p": p})
        .group_by("row").agg(pl.all())
    )

    dataset = (
        dataset
        .sort(owner_col, entry_col)
        .group_by(owner_col).agg(pl.all())
    )

    domain_records = (
        domain_records
        .sort(record_col, entry_col)
        .group_by(record_col).agg(pl.all())
    )

    rng = np.random.default_rng()
    targets = rng.uniform(0, 1, size=dataset.height)
    target_expr = pl.lit(targets).alias("target")
    
    record_attrs = [c for c in dataset.collect_schema() if c != owner_col]
    cdf_expr = pl.col("p").list.eval(pl.element().cum_sum()).alias("cdf")
    
    choice_args_expr = pl.concat_list("cdf", "target").alias("choice_args")
    cdf_arg_expr = pl.element().head(pl.element().len() - 1)
    target_arg_expr = pl.element().get(-1)

    choice_expr = (
        pl.col("choice_args")
        .list.eval(cdf_arg_expr.search_sorted(target_arg_expr, side="right").get(0))
        .list.item()
        .alias("choice")
    )
    
    return (
        dataset
        .join(domain_records, on=record_attrs) # Get record id
        .select(owner_col, record_col)
        .join(m_df, left_on=record_col, right_on="row") # Get probabilities
        .with_columns(cdf_expr, target_expr)
        .select(owner_col, pl.col("col").alias(record_col), choice_args_expr)
        .select(owner_col, record_col, choice_expr)
        .select(owner_col, pl.col(record_col).list.get("choice"))
        .join(domain_records, on=record_col).drop(record_col)
        .explode(pl.exclude(owner_col))
    )

Now we can generate different sanitised datasets, applying geometric noise, and see how risk varies:

In [41]:
m_amount = partial(mechanism.geometric, alpha=1/3)
m = mechanism.record(input_domain, amount=m_amount)
sanitised_dataset = run_mechanism(dataset, input_domain, m)
adv_model = model.generic(pi, input_domain, m, dataset, sanitised_dataset, hint)

measure.linkage_risk(adv_model)

np.float64(0.16666666666666666)